# Entrega 3 - Modelo, metricas y preprocesamiento

Este notebook ejecuta el pipeline reproducible de semana 7 usando los modulos en `src/`. No crea formulas analiticas nuevas; compara estructuras de datos para Tableau con metricas de validacion estructural.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
PROJECT_ROOT


## 1. Carga de datos

El archivo esperado es `data/raw/all_ai_models.csv`.


In [ ]:
from src.config import RAW_DATASET
from src.io_utils import load_csv

raw_df = load_csv(RAW_DATASET)
raw_df.shape


## 2. Preprocesamiento reproducible


In [ ]:
from src.preprocessing import clean_ai_models, build_quality_summary

clean_df = clean_ai_models(raw_df)
quality_summary = build_quality_summary(raw_df, clean_df)
display(quality_summary)
clean_df.head()


## 3. Comparacion de opciones de modelo

Se comparan dos opciones: tabla plana y esquema estrella. Las metricas verifican preservacion de filas, unicidad de modelos, nulos criticos y llaves relacionales.


In [ ]:
from src.modeling import compare_model_options

model_comparison = compare_model_options(clean_df)
display(model_comparison)


## 4. Exportacion para Tableau


In [ ]:
from src.config import CLEAN_DATASET, OUTPUTS_REPORTS, OUTPUTS_TABLEAU
from src.io_utils import save_csv
from src.modeling import build_flat_model, build_star_schema

save_csv(clean_df, CLEAN_DATASET)
save_csv(quality_summary, OUTPUTS_REPORTS / 'quality_summary.csv')
save_csv(model_comparison, OUTPUTS_REPORTS / 'model_options_comparison.csv')

for name, table in build_flat_model(clean_df).items():
    save_csv(table, OUTPUTS_TABLEAU / f'{name}.csv')

for name, table in build_star_schema(clean_df).items():
    save_csv(table, OUTPUTS_TABLEAU / f'{name}.csv')

print('Archivos exportados en:', OUTPUTS_TABLEAU)


## 5. Decision metodologica

La recomendacion se documenta en `docs/entrega_3_modelo_metricas_preprocesamiento.md`. En terminos operativos, se elige el esquema estrella cuando conserva las filas del dataset limpio, mantiene unicidad por `Model` y no genera llaves relacionales nulas.
